# LinkedIn Zip Solver — Setup & Run Notebook (v2)

This notebook provisions the environment, verifies all modules, trains the CNN, runs 5 solver tests, and launches the live LinkedIn Zip solver.

**What's new in v2:**
- DOM-first vision (reads `[data-cell-idx]` directly — pixel-perfect, no OCR)
- Color-based wall detection (beige walls vs white active cells)
- Optimized solver (Warnsdorff heuristic + deadlock detection + CNN prior)
- Trained CNN weights shipped as `zip_cnn.pth`
- Fixed the `'str' // int` crash from v1 via universal type coercion

## 1. Environment Check

In [1]:
import sys
import os
from pathlib import Path

repo_root = Path.cwd()
sys.path.insert(0, str(repo_root))

print('Python:', sys.executable)
print('Version:', sys.version)
print('Repo root:', repo_root)
print()

# Check installed packages.
packages = ['playwright', 'torch', 'torchvision', 'cv2', 'numpy', 'PIL']
for pkg in packages:
    try:
        mod = __import__(pkg)
        ver = getattr(mod, '__version__', 'ok')
        print(f'  {pkg:15s} {ver}')
    except ImportError:
        print(f'  {pkg:15s} MISSING')

Python: c:\Users\shiva\Documents\Zip\linkedin_zip_solver\.venv\Scripts\python.exe
Version: 3.14.2 (tags/v3.14.2:df79316, Dec  5 2025, 17:18:21) [MSC v.1944 64 bit (AMD64)]
Repo root: c:\Users\shiva\Documents\Zip\linkedin_zip_solver

  playwright      ok
  torch           2.12.1+cpu
  torchvision     0.27.1+cpu
  cv2             5.0.0
  numpy           2.5.1
  PIL             12.3.0


## 2. Install Missing Dependencies

If any packages are missing, this cell installs them automatically.

In [2]:
import subprocess

requirements = [
    'playwright>=1.40.0',
    'torch>=2.1.0',
    'torchvision>=0.16.0',
    'opencv-python>=4.8.0',
    'numpy>=1.24.0',
    'pillow>=10.0.0',
]

missing = []
for req in requirements:
    pkg = req.split('>=')[0]
    try:
        __import__(pkg if pkg != 'opencv-python' else 'cv2')
    except ImportError:
        missing.append(req)

if missing:
    print('Installing missing packages:', missing)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install'] + missing)
    print('Done. Please restart the kernel and re-run this cell.')
else:
    print('All dependencies are installed.')

# Also ensure Playwright's Chromium is installed.
try:
    subprocess.check_call([sys.executable, '-m', 'playwright', 'install', 'chromium'])
except Exception as e:
    print('Playwright install note:', e)

Installing missing packages: ['pillow>=10.0.0']
Done. Please restart the kernel and re-run this cell.


## 3. Import All Modules

Verify that every module loads without errors.

In [3]:
import main
import model
import solver
import vision
import executor
import train
import setup_env
import package

print('All modules imported successfully:')
print('  vision, solver, model, executor, train, setup_env, package, main')

ModuleNotFoundError: No module named 'setup_env'

## 4. CNN Model Smoke Test

Load the trained CNN (if weights exist) and run a single inference.

In [ ]:
from model import load_model, NUM_CHANNELS, HAS_TORCH
import torch

print(f'PyTorch available: {HAS_TORCH}')

if HAS_TORCH:
    model_obj = load_model()
    if model_obj is not None:
        params = sum(p.numel() for p in model_obj.parameters())
        print(f'Model loaded: {params:,} parameters')
        
        # Dummy inference.
        dummy = torch.zeros(1, NUM_CHANNELS, 8, 8)
        policy, value = model_obj.predict(dummy)
        print(f'Policy probabilities: {policy.tolist()}')
        print(f'Value score: {value:.4f}')
    else:
        print('Model not loaded (torch available but load failed).')
else:
    print('torch not installed — solver will use plain DFS.')

PyTorch available: True
Model loaded: 95,464 parameters
Policy probabilities: [0.015581503510475159, 0.4423932731151581, 0.023964038118720055, 0.5180611610412598]
Value score: -0.8955


## 5. Solver Test — 5 Synthetic Boards

Run the solver on 5 randomly generated boards and verify each solution is correct.

In [ ]:
import time
import random
import numpy as np
from vision import generate_synthetic_state
from solver import ZipSolver, SolverConfig
from executor import validate_path, build_drag_path, ExecutorConfig

config = SolverConfig(
    max_iterations=2_000_000,
    use_cnn_prior=True,
    use_heuristic_prior=True,
    use_deadlock_check=True,
    verbose=False,
)

all_passed = True
for seed in range(5):
    state, true_path = generate_synthetic_state(rows=6, cols=6, seed=seed)
    solver_inst = ZipSolver(state, model=model_obj, config=config)
    t0 = time.time()
    path = solver_inst.solve()
    elapsed = time.time() - t0
    
    if path is None:
        print(f'Seed {seed}: FAIL — no solution')
        all_passed = False
        continue
    
    ok = (validate_path(path) and
          path[0] == state.start and
          path[-1] == state.goal and
          len(set(path)) == len(path) and
          len(path) == len(true_path))
    
    status = 'PASS' if ok else 'FAIL'
    if not ok:
        all_passed = False
    print(f'Seed {seed}: {status} — {len(path)} cells, '
          f'{solver_inst.nodes_explored} nodes, {elapsed:.3f}s')

print(f'\nSolver test: {"ALL PASSED" if all_passed else "SOME FAILED"}')

Seed 0: PASS — 21 cells, 21 nodes, 0.025s
Seed 1: PASS — 20 cells, 20 nodes, 0.010s
Seed 2: PASS — 23 cells, 23 nodes, 0.010s
Seed 3: PASS — 23 cells, 55 nodes, 0.023s
Seed 4: PASS — 20 cells, 20 nodes, 0.008s

Solver test: ALL PASSED


## 6. DOM Parsing Test

Test the DOM-based vision parser on a synthetic DOM state (simulates what `page.evaluate()` returns from LinkedIn).

In [ ]:
from vision import build_state_from_dom, CellType

# Build a synthetic 4x4 DOM with walls.
rows, cols = 4, 4
numbered = {0: 1, 3: 2, 15: 3}  # start, mid, goal
wall_indices = {5, 7, 12}
active_bg = 'rgb(255, 255, 255)'
wall_bg = 'rgb(230, 220, 200)'

cells_data = []
for r in range(rows):
    for c in range(cols):
        idx = r * cols + c
        if idx in wall_indices:
            bg, num = wall_bg, None
        elif idx in numbered:
            bg, num = active_bg, numbered[idx]
        else:
            bg, num = active_bg, None
        cells_data.append({
            'idx': idx, 'number': num,
            'ariaLabel': str(num) if num else None,
            'bg': bg, 'x': c * 50, 'y': r * 50, 'w': 50, 'h': 50,
        })

dom_state = {
    'count': rows * cols,
    'cells': cells_data,
    'gridRect': {'x': 0, 'y': 0, 'w': cols * 50, 'h': rows * 50},
}

state = build_state_from_dom(dom_state)
print(f'Parsed grid ({state.shape[0]}x{state.shape[1]}):')
for r in range(rows):
    row_str = '  '
    for c in range(cols):
        ct = state.grid[r, c]
        if ct == int(CellType.WALL):
            row_str += ' W '
        elif ct == int(CellType.START):
            row_str += ' S '
        elif ct == int(CellType.GOAL):
            row_str += ' G '
        elif ct == int(CellType.NUMBERED):
            for cell in state.cells:
                if cell.row == r and cell.col == c:
                    row_str += f' {cell.number} '
                    break
        else:
            row_str += ' . '
    print(row_str)

print(f'Start: {state.start}, Goal: {state.goal}')
print(f'Waypoints: {state.numbers.tolist()}')
print(f'Walls detected: {sum(1 for r in range(rows) for c in range(cols) if state.grid[r,c] == int(CellType.WALL))}')

Parsed grid (4x4):
   S  .  .  2 
   .  W  .  W 
   .  .  .  . 
   W  .  .  G 
Start: (0, 0), Goal: (3, 3)
Waypoints: [1, 2, 3]
Walls detected: 3


## 7. Train the CNN (Optional)

If `zip_cnn.pth` doesn't exist or you want to retrain, run this cell. It generates 200 synthetic games and trains for 15 epochs (~1-2 minutes on CPU).

In [ ]:
from pathlib import Path
weights_path = Path('zip_cnn.pth')
if weights_path.exists():
    print(f'Weights already exist: {weights_path} ({weights_path.stat().st_size / 1024:.1f} KB)')
    print('To retrain, delete the file and re-run this cell.')
else:
    print('Training CNN...')
    import subprocess
    result = subprocess.run(
        [sys.executable, 'train.py', '--num-games', '200', '--epochs', '15'],
        cwd=str(repo_root), text=True, capture_output=True
    )
    print(result.stdout)
    if result.stderr:
        print('STDERR:', result.stderr)
    if weights_path.exists():
        print(f'Training complete: {weights_path.stat().st_size / 1024:.1f} KB')

Weights already exist: zip_cnn.pth (386.3 KB)
To retrain, delete the file and re-run this cell.


## 8. Run the Live Solver

Opens Chrome, navigates to LinkedIn Zip, extracts the board, solves it, and drags the solution.

**First run:** A visible Chrome window opens so you can log into LinkedIn. Cookies persist in `.browser_data/` for subsequent runs.

**Note:** If you get a vision error, make sure:
1. You are logged into LinkedIn in the Chrome window
2. The Zip game board is fully loaded and visible
3. No splash screen / modal is covering the board

In [4]:
import subprocess

print('Starting the live solver...')
print('A Chrome window will open. If prompted, log into LinkedIn.')
print()

result = subprocess.run(
    [sys.executable, 'main.py', '--solve-only', '--skip-setup'],
    cwd=str(repo_root),
    text=True,
    capture_output=False
)
print(f'\nExit code: {result.returncode}')

Starting the live solver...
A Chrome window will open. If prompted, log into LinkedIn.


Exit code: 0


## 9. Package the Project

Builds `dist/linkedin_zip_solver.zip` containing all source files, trained weights, requirements, and this notebook.

In [ ]:
result = subprocess.run(
    [sys.executable, 'package.py'],
    cwd=str(repo_root),
    text=True,
    capture_output=True
)
print(result.stdout)
if result.stderr:
    print('STDERR:', result.stderr)
print(f'Exit code: {result.returncode}')

zip_path = repo_root / 'dist' / 'linkedin_zip_solver.zip'
if zip_path.exists():
    print(f'\nArchive: {zip_path} ({zip_path.stat().st_size / 1024:.1f} KB)')

[package]   + vision.py
[package]   + solver.py
[package]   + model.py
[package]   + executor.py
[package]   + setup_env.py
[package]   + package.py
[package]   + main.py
[package]   + train.py
[package]   + requirements.txt
[package]   + README.md
[package]   + VENV_SETUP.md
[package]   + setup_notebook.ipynb
[package]   + zip_cnn.pth (386.3 KB)
[package] Creating archive: C:\Users\shiva\Documents\Zip\linkedin_zip_solver\dist\linkedin_zip_solver.zip
[package] Archive size: 383.1 KB
[package] Done: C:\Users\shiva\Documents\Zip\linkedin_zip_solver\dist\linkedin_zip_solver.zip

Exit code: 0

Archive: c:\Users\shiva\Documents\Zip\linkedin_zip_solver\dist\linkedin_zip_solver.zip (383.1 KB)
